# Phase 3: multiclass intrusion classification
Compare recorded Logistic Regression and Random Forest results for 15 labels. This notebook does not train models or rescore test. Run `python -m src.train_multiclass` first if needed, and choose the NetworkMonitoring (Python 3.12) kernel.

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise RuntimeError("Open Jupyter in the project root or notebooks directory")
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from IPython.display import display, Markdown, Image
from src.predict import predict_frame
path = ROOT / "reports/multiclass_baseline_metrics.json"
if not path.exists():
    raise RuntimeError("Run python -m src.train_multiclass first")
results = json.loads(path.read_text())

## 1. Experiment design
Training only is used for fitting. Validation macro F1 chooses the model; lower benign FPR breaks ties. The selected model is tested without refitting on validation. Predictions use the highest-scoring class. **The test split is the same benchmark used for binary evaluation, not a new independent test.**

In [ ]:
print("Sizes:", results["split_records"])
print("Selection:", results["selection_rule"])
print("Selected model:", results["selected_model"])
print("Test scope:", results["test_scope"])
display(pd.Series(results["classes"], name="label").to_frame())

## 2. Validation comparison
Macro F1 weights each saved class equally. Weighted F1 weights by support. Both are needed because small classes can be poorly learned despite high accuracy.

In [ ]:
keys = ["accuracy", "macro_f1", "weighted_f1", "false_positive_rate"]
comparison = pd.DataFrame({name: {k: item["metrics"][k] for k in keys} for name, item in {**results["candidates"], "majority_dummy": results["dummy"]}.items()}).T
display(comparison)
comparison[["macro_f1", "weighted_f1"]].plot.bar(figsize=(9, 4))
plt.ylim(0, 1.05)
plt.ylabel("Validation F1")
plt.tight_layout()
plt.show()
for name, item in results["candidates"].items():
    print(name, "converged:", item["converged"], "fit seconds:", round(item["fit_seconds"], 2), "warnings:", item["warnings"])

## 3. Recorded test performance by class
Read precision, recall, and F1 together with support. A perfect score on one record is not reliable evidence of generalization. A support below 20 is flagged descriptively; 20 is not a statistical reliability cutoff.

In [ ]:
display(pd.Series({k: results["test"][k] for k in keys}, name="test"))
per_class = pd.DataFrame({c: results["test"]["classification_report"][c] for c in results["classes"]}).T
per_class["support"] = per_class["support"].astype(int)
per_class["small_holdout"] = per_class["support"] < 20
display(per_class)
per_class["f1-score"].sort_values().plot.barh(figsize=(10, 6))
plt.xlim(0, 1.05)
plt.xlabel("Recorded test F1")
plt.tight_layout()
plt.show()

## 4. Confusion matrices and common errors
Rows are true labels; columns are predicted labels. The normalized matrix shows per-class recall distribution. These are descriptive test findings, not permission to tune on test.

In [ ]:
display(Image(filename=str(ROOT / "reports/multiclass_test_confusion_matrix.png")))
display(pd.DataFrame(results["test"]["top_confusions"]))
print("Rare test classes:", results["rare_test_classes"])

## 5. Offline multiclass inference
Use validation examples for this demonstration. Only feature columns enter the pipeline. Scores are uncalibrated; they are not guaranteed real-network probabilities.

In [ ]:
artifact = joblib.load(ROOT / "models/multiclass_detector.joblib")
validation = pd.read_parquet(ROOT / "data/processed/phase2/validation_raw.parquet")
examples = validation.groupby("Label", sort=True).head(1)
predictions = predict_frame(examples, artifact)
assert set(predictions["prediction"]).issubset(set(results["classes"]))
display(pd.concat([examples[["Label"]], predictions], axis=1))

## 6. Report and next phase
Binary and multiclass baselines are complete. Live-network generalization is untested. Future work should establish compatible flow extraction and PCAP replay before packet monitoring.

In [ ]:
display(Markdown((ROOT / "reports/phase3_multiclass_baseline.md").read_text()))